# RF branch — clean final pipeline (train → test → fusion export)

What this notebook fixes compared with the old one:
1. **One dataset path** (auto-detected) and **one output folder** (`RF_Models_v2`, old files are never overwritten).
2. **Batches are shuffled across the whole dataset.** The old shards were almost single-class (17 of 18 train shards were pure Drone or pure Background), so almost every training batch contained only one class.
3. **One preprocessing function for every split** (bicubic, uint8, no ImageNet normalisation). The old shards used bilinear, the old test loader used bicubic.
4. **Deterministic, sorted file order, and paths are saved** with every embedding, so RF can be joined to audio and video.
5. **Train, val and test** probabilities and embeddings are exported in one file, with a recording ID and segment index.
6. **Leakage audit** (recordings that appear in more than one split) and an optional recording-grouped split.
7. Checkpoint chosen by **val macro-F1** with early stopping, and the label convention is explicit: **1 = Drone** everywhere in the new files.

In [ ]:
# ============================================================
# 1. SETUP  (edit only the block marked EDIT)
# ============================================================
import os, sys, re, json, random, time
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# ---------------- EDIT ----------------
RF_ROOT_CANDIDATES = [
    "/content/drive/MyDrive/RF_Spectrograms",            # your layout
    "/content/drive/MyDrive/dataset/RF_Spectrograms",    # old layout, kept as a fallback
]
OUT_DIR         = "/content/drive/MyDrive/RF_Models_v2"  # everything new goes here
SPLIT_MODE      = "official"   # "official" = folders exactly as given (comparable with TRIDENT)
                               # "grouped"  = every recording lives in ONE split only (no leakage)
GROUP_SPLIT_CSV = None         # optional: csv with columns  recording,split  (e.g. made from the audio manifest)
EPOCHS, BATCH, LR, WD, PATIENCE = 12, 64, 1e-4, 1e-4, 4
SEED = 42
# --------------------------------------

RF_ROOT = next((p for p in RF_ROOT_CANDIDATES if os.path.isdir(p)), None)
if RF_ROOT is None:
    print("Could not find the dataset. Top-level folders in MyDrive:")
    print(sorted(os.listdir("/content/drive/MyDrive")))
    raise FileNotFoundError("Fix RF_ROOT_CANDIDATES above.")
os.makedirs(OUT_DIR, exist_ok=True)
CACHE_NPY = os.path.join(OUT_DIR, "rf_cache_uint8.npy")   # all images, 224x224, uint8 (saved once, reused every session)
INDEX_CSV = os.path.join(OUT_DIR, "rf_index.csv")
print("RF_ROOT :", RF_ROOT)
print("OUT_DIR :", OUT_DIR)

random.seed(SEED); np.random.seed(SEED)

In [ ]:
# ============================================================
# 2. TRIDENT code + GPU
# ============================================================
import torch, torch.nn as nn
if not os.path.isdir("/content/TRIDENT"):
    !git clone -q https://github.com/TRIDENT-2025/TRIDENT.git /content/TRIDENT
for d in ["models", "models/resnet", "dataset", "training", "utils"]:
    p = f"/content/TRIDENT/{d}"
    if os.path.isdir(p):
        open(os.path.join(p, "__init__.py"), "a").close()
sys.path.insert(0, "/content/TRIDENT")
from models.get_model import get_model

torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU: Runtime > Change runtime type)")

In [ ]:
# ============================================================
# 3. INDEX every image (sorted, so the order is the same on every machine)
#    layout: <split>/<Drone|Background>/<Drone_k|Background_k>/<Images|Images_Spectrograms>/<file>
# ============================================================
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def build_index(root):
    rows = []
    for split in ("Train", "Validation", "Test"):
        for cls in ("Drone", "Background"):
            base = os.path.join(root, split, cls)
            if not os.path.isdir(base):
                print("missing folder:", base); continue
            for dp, dns, fns in os.walk(base):
                dns.sort()
                for fn in sorted(fns):
                    if fn.lower().endswith(IMG_EXT):
                        rel = os.path.relpath(os.path.join(dp, fn), root)
                        parts = rel.split(os.sep)
                        nums = re.findall(r"\d+", os.path.splitext(fn)[0])
                        rows.append(dict(
                            path=rel, split_official=split, cls=cls,
                            recording=parts[2],                 # e.g. Drone_13  (same naming as the audio files)
                            filename=fn,
                            seg_idx=int(nums[-1]) if nums else -1,
                            y_drone=1 if cls == "Drone" else 0))   # 1 = Drone in every NEW file
    return pd.DataFrame(rows)

if os.path.exists(INDEX_CSV):
    df = pd.read_csv(INDEX_CSV)
    print("Loaded index:", INDEX_CSV)
else:
    df = build_index(RF_ROOT)
    df.to_csv(INDEX_CSV, index=False)
print(df.groupby(["split_official", "cls"]).size().unstack())
print("\nexample rows:\n", df.sample(5, random_state=0)[["path", "recording", "filename", "seg_idx"]].to_string(index=False))

In [ ]:
# ============================================================
# 4. DECODE ONCE -> uint8 cache (the slow Drive read happens only the first time)
#    ONE preprocessing for all splits: RGB, 224x224 BICUBIC, kept as uint8, divided by 255 at use time. No ImageNet mean/std.
# ============================================================
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

def load_uint8(rel_path):
    im = Image.open(os.path.join(RF_ROOT, rel_path)).convert("RGB").resize((224, 224), Image.BICUBIC)
    return np.asarray(im, dtype=np.uint8)            # H, W, C

if os.path.exists(CACHE_NPY):
    X = np.load(CACHE_NPY)
    assert len(X) == len(df), "cache and index disagree -> delete rf_cache_uint8.npy and rf_index.csv and rerun"
    print("Loaded cache:", X.shape)
else:
    X = np.empty((len(df), 224, 224, 3), dtype=np.uint8)
    with ThreadPoolExecutor(8) as ex:
        for i, arr in enumerate(tqdm(ex.map(load_uint8, df["path"]), total=len(df))):
            X[i] = arr
    np.save(CACHE_NPY, X)
    print("Saved cache:", X.shape, f"({X.nbytes/1e9:.2f} GB)")

In [ ]:
# ============================================================
# 5. SPLITS + leakage audit
# ============================================================
def make_grouped_split(df, seed=SEED, frac=(0.6, 0.2, 0.2)):
    """Whole recordings go to ONE split. Stratified by class."""
    rng = np.random.RandomState(seed)
    out = {}
    for cls in ("Drone", "Background"):
        recs = sorted(df.loc[df.cls == cls, "recording"].unique())
        rng.shuffle(recs)
        n = len(recs); a = int(round(frac[0]*n)); b = a + int(round(frac[1]*n))
        for i, r in enumerate(recs):
            out[r] = "Train" if i < a else ("Validation" if i < b else "Test")
    return out

if SPLIT_MODE == "official":
    df["split"] = df["split_official"]
elif SPLIT_MODE == "grouped":
    if GROUP_SPLIT_CSV:
        m = pd.read_csv(GROUP_SPLIT_CSV).set_index("recording")["split"].to_dict()
    else:
        m = make_grouped_split(df)
        pd.Series(m, name="split").rename_axis("recording").to_csv(os.path.join(OUT_DIR, "rf_recording_split.csv"))
    df["split"] = df["recording"].map(m)
    assert df["split"].notna().all(), "some recordings are missing from the split csv"
else:
    raise ValueError("SPLIT_MODE must be 'official' or 'grouped'")

n_rec = df.groupby("recording")["split"].nunique()
for s_ in ("Train", "Validation", "Test"):
    assert set(df.loc[df.split == s_, "cls"]) == {"Drone", "Background"}, f"{s_} is missing a class -> change SEED or pass GROUP_SPLIT_CSV"
print(f"SPLIT_MODE = {SPLIT_MODE}")
print(f"Recordings appearing in more than one split: {(n_rec > 1).sum()} of {len(n_rec)}  (0 means no leakage)")
print(df.groupby(["split", "cls"]).size().unstack(), "\n")

y_all = df["y_drone"].values.astype(np.float32)
idx = {s: np.where(df["split"].values == s)[0] for s in ("Train", "Validation", "Test")}

In [ ]:
# ============================================================
# 6. MODEL, batching (global shuffle!), metrics
# ============================================================
def new_model():
    return get_model("resnet10", n_classes=2, m=1).to(DEVICE)   # outputs[4] = 2048-d embedding, outputs[5] = sigmoid score

def to_batch(ids, augment=False):
    # .contiguous() matters: TRIDENT's forward() uses x.view(...), which fails on a permuted (non-contiguous) tensor
    x = torch.from_numpy(X[ids]).permute(0, 3, 1, 2).contiguous().to(DEVICE).float().div_(255.0)   # B,3,224,224 in [0,1]
    if augment:                                                                # light, orientation-agnostic
        x = x * (1 + 0.1 * (torch.rand(len(x), 1, 1, 1, device=DEVICE) * 2 - 1)) + 0.02 * torch.randn_like(x)
        x = x.clamp_(0, 1)
    return x.unsqueeze(1).contiguous()                                         # B,T=1,3,224,224

def iterate(ids, bs, shuffle, augment=False):
    ids = np.random.permutation(ids) if shuffle else ids      # <- the whole split is shuffled, so every batch mixes classes
    for s in range(0, len(ids), bs):
        b = np.sort(ids[s:s+bs])
        yield b, to_batch(b, augment), torch.from_numpy(y_all[b]).to(DEVICE)

@torch.no_grad()
def predict(model, ids, bs=64, want_emb=False):
    model.eval(); P, E = [], []
    for b, xb, _ in iterate(ids, bs, shuffle=False):
        out = model(xb)
        P.append(out[5].reshape(-1).float().cpu())                  # reshape(-1): safe even if the last batch has 1 sample
        if want_emb: E.append(out[4].float().cpu())
    p = torch.cat(P).numpy()
    return (p, torch.cat(E).numpy()) if want_emb else p

from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, roc_auc_score
def metrics(y, p, thr=0.5):
    yh = (p >= thr).astype(int); y = y.astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    return dict(acc=accuracy_score(y, yh), macro_f1=f1_score(y, yh, average="macro"),
                drone_prec=precision_score(y, yh, zero_division=0), drone_rec=recall_score(y, yh, zero_division=0),
                miss_rate=fn/max(fn+tp, 1), false_alarm=fp/max(fp+tn, 1),
                auc=roc_auc_score(y, p) if len(set(y)) > 1 else float("nan"), tn=tn, fp=fp, fn=fn, tp=tp)

In [ ]:
# ============================================================
# 7. TRAIN  (best epoch = best validation macro-F1, early stopping)
# ============================================================
CKPT_BEST = os.path.join(OUT_DIR, "rf_resnet10_v2_best.pth")
CKPT_LAST = os.path.join(OUT_DIR, "rf_resnet10_v2_last.pth")

model = new_model()
criterion = nn.BCELoss()
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
hist, best_f1, bad = [], -1.0, 0

for ep in range(1, EPOCHS + 1):
    t0 = time.time(); model.train(); tl = tc = tn_ = 0
    for b, xb, yb in iterate(idx["Train"], BATCH, shuffle=True, augment=True):
        if len(b) < 2:            # BatchNorm cannot train on a single sample
            continue
        p = model(xb)[5].reshape(-1)
        loss = criterion(p, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        tl += loss.item() * len(b); tc += ((p >= 0.5).float() == yb).sum().item(); tn_ += len(b)
    pv = predict(model, idx["Validation"]); mv = metrics(y_all[idx["Validation"]], pv)
    vloss = float(nn.functional.binary_cross_entropy(torch.tensor(pv).clamp(1e-6, 1-1e-6), torch.tensor(y_all[idx["Validation"]])))
    hist.append(dict(epoch=ep, train_loss=tl/tn_, train_acc=tc/tn_, val_loss=vloss, val_acc=mv["acc"], val_macro_f1=mv["macro_f1"],
                     val_miss=mv["miss_rate"], val_false_alarm=mv["false_alarm"]))
    print(f"ep {ep:02d} | train loss {tl/tn_:.4f} acc {tc/tn_:.4f} | val loss {vloss:.4f} acc {mv['acc']:.4f} "
          f"macroF1 {mv['macro_f1']:.4f} miss {mv['miss_rate']:.3f} FA {mv['false_alarm']:.3f} | {time.time()-t0:.0f}s")
    torch.save(dict(model_state_dict=model.state_dict(), epoch=ep, convention="p_drone"), CKPT_LAST)
    if mv["macro_f1"] > best_f1:
        best_f1, bad = mv["macro_f1"], 0
        torch.save(dict(model_state_dict=model.state_dict(), epoch=ep, val_macro_f1=best_f1, convention="p_drone",
                        split_mode=SPLIT_MODE), CKPT_BEST)
        print("   ✓ new best saved")
    else:
        bad += 1
        if bad >= PATIENCE:
            print("early stopping"); break

pd.DataFrame(hist).to_csv(os.path.join(OUT_DIR, "rf_train_history.csv"), index=False)
print("best val macro-F1:", round(best_f1, 4))

In [ ]:
# ============================================================
# 8. TEST (never used for training or checkpoint choice) + curves
# ============================================================
import matplotlib.pyplot as plt
ck = torch.load(CKPT_BEST, map_location=DEVICE)
model = new_model(); model.load_state_dict(ck["model_state_dict"]); model.eval()
print("loaded epoch", ck["epoch"])

res = {}
for s in ("Train", "Validation", "Test"):
    res[s] = metrics(y_all[idx[s]], predict(model, idx[s]))
tab = pd.DataFrame(res).T[["acc", "macro_f1", "drone_prec", "drone_rec", "miss_rate", "false_alarm", "auc", "tn", "fp", "fn", "tp"]]
print(tab.round(4).to_string())
tab.to_csv(os.path.join(OUT_DIR, "rf_results.csv"))

h = pd.DataFrame(hist)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(h.epoch, h.train_acc, label="train"); ax[0].plot(h.epoch, h.val_acc, label="val"); ax[0].set_title("accuracy"); ax[0].legend()
ax[1].plot(h.epoch, h.train_loss, label="train"); ax[1].plot(h.epoch, h.val_loss, label="val"); ax[1].set_title("BCE loss"); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "rf_curves.png"), dpi=150); plt.show()

In [ ]:
# ============================================================
# 9. FUSION EXPORT: one file, every split, with IDs
#    p_drone = probability of Drone (1 = Drone).  emb = 2048-d.
# ============================================================
p_all, e_all = predict(model, np.arange(len(df)), want_emb=True)
EXPORT = os.path.join(OUT_DIR, "rf_fusion_export.npz")
np.savez(EXPORT,
         path=df["path"].values.astype(str), recording=df["recording"].values.astype(str),
         filename=df["filename"].values.astype(str), seg_idx=df["seg_idx"].values,
         split=df["split"].values.astype(str), split_official=df["split_official"].values.astype(str),
         y_drone=df["y_drone"].values, p_drone=p_all.astype(np.float32), emb=e_all.astype(np.float32),
         meta=json.dumps(dict(convention="1=Drone", split_mode=SPLIT_MODE, ckpt=os.path.basename(CKPT_BEST), epoch=int(ck["epoch"]))))

z = np.load(EXPORT, allow_pickle=True)
print({k: z[k].shape for k in z.files if k != "meta"})
print("meta:", z["meta"])
for s in ("Train", "Validation", "Test"):
    m = z["split"] == s
    print(f"{s:10s} n={m.sum():5d}  mean p_drone | Drone={z['p_drone'][m & (z['y_drone']==1)].mean():.3f}  Background={z['p_drone'][m & (z['y_drone']==0)].mean():.3f}")
print("\nfirst filenames (check how segment numbers map to the audio windows):")
print(df[["recording", "filename", "seg_idx"]].head(8).to_string(index=False))

In [ ]:
# ============================================================
# 10. (optional) why did the OLD model drop from val 89.8% to a worse background recall on test?
#     Old shards were resized with torchvision's default (BILINEAR); the old test loader used BICUBIC.
#     This re-scores the OLD checkpoint on the test images with both, to see if the interpolation matters.
# ============================================================
OLD_CKPT = "/content/drive/MyDrive/RF_Models/rf_resnet10_corrected_best.pth"     # legacy: output = P(Background)
if os.path.exists(OLD_CKPT):
    old = get_model("resnet10", n_classes=2, m=1).to(DEVICE)
    sd = torch.load(OLD_CKPT, map_location=DEVICE); old.load_state_dict(sd.get("model_state_dict", sd)); old.eval()
    tid = idx["Test"] if SPLIT_MODE == "official" else np.where(df["split_official"].values == "Test")[0]
    for name, interp in [("BILINEAR", Image.BILINEAR), ("BICUBIC", Image.BICUBIC)]:
        P = []
        with torch.no_grad():
            for s in range(0, len(tid), 64):
                ims = [np.asarray(Image.open(os.path.join(RF_ROOT, df.path[i])).convert("RGB").resize((224, 224), interp), dtype=np.uint8) for i in tid[s:s+64]]
                xb = torch.from_numpy(np.stack(ims)).permute(0, 3, 1, 2).contiguous().to(DEVICE).float().div_(255).unsqueeze(1).contiguous()
                P.append((1 - old(xb)[5].reshape(-1)).cpu())          # legacy output is P(Background) -> flip
        m = metrics(y_all[tid], torch.cat(P).numpy())
        print(f"{name:9s} acc {m['acc']:.4f}  miss {m['miss_rate']:.3f}  false-alarm {m['false_alarm']:.3f}")
else:
    print("old checkpoint not found, skipping")